<a href="https://colab.research.google.com/github/CynthiahL/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/agent_cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%%writefile cleaning_pipeline.py
import pandas as pd
import numpy as np


def clean_agent_data(txns: pd.DataFrame, agents: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Audits, cleans, flags, and merges mobile-money agent transaction data.

    Parameters
    ----------
    txns : pd.DataFrame
        Raw transaction extract DataFrame (agent_transactions.csv).
    agents : pd.DataFrame
        Agent master register DataFrame (agents.csv).

    Returns
    -------
    tuple[pd.DataFrame, dict]
        A tuple containing the cleaned, merged DataFrame and a dictionary
        logging row counts and cleaning actions at each stage.
    """
    df = txns.copy()
    log = {"initial_rows": len(df)}

    # 1. Uniqueness: remove duplicate txn_id rows, keeping first original record
    dup_count = int(df.duplicated(subset=["txn_id"]).sum())
    df = df.drop_duplicates(subset=["txn_id"], keep="first").copy()
    log["duplicates_removed"] = dup_count

    # 2. Consistency: standardise txn_type to 'Deposit' and 'Withdrawal'
    df["txn_type"] = df["txn_type"].astype("string").str.strip().str.title()
    invalid_type_mask = df["txn_type"].isna() | (~df["txn_type"].isin(["Deposit", "Withdrawal"]))
    log["invalid_txn_types"] = int(invalid_type_mask.sum())

    # 3. Validity & types: normalise date delimiters and convert amount/txn_date
    df["txn_date"] = pd.to_datetime(
        df["txn_date"].astype("string").str.replace("/", "-", regex=False),
        format="%Y-%m-%d",
        errors="coerce",
    )
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

    # 4. Completeness: flag missing amounts and impute with median
    df["amount_imputed"] = df["amount"].isna()
    log["amounts_imputed"] = int(df["amount_imputed"].sum())
    median_amt = float(df["amount"].median())
    df["amount"] = df["amount"].fillna(median_amt)
    log["imputation_median_kes"] = median_amt

    # 5. Outliers: IQR method (Q3 + 1.5 * IQR) to flag suspicious amounts
    q1 = df["amount"].quantile(0.25)
    q3 = df["amount"].quantile(0.75)
    iqr = q3 - q1
    lower_fence = max(0.0, q1 - 1.5 * iqr)
    upper_fence = q3 + 1.5 * iqr
    df["is_outlier"] = (df["amount"] < lower_fence) | (df["amount"] > upper_fence)
    log["iqr_upper_fence"] = round(float(upper_fence), 2)
    log["outliers_flagged"] = int(df["is_outlier"].sum())

    # 6. Merge: left join onto agent register and label unknown agents
    clean_df = df.merge(agents, on="agent_id", how="left")
    log["unknown_agent_txns"] = int(clean_df["county"].isna().sum())
    clean_df["county"] = clean_df["county"].fillna("Unknown")

    log["final_rows"] = len(clean_df)
    return clean_df, log


Writing cleaning_pipeline.py


In [4]:
import os
import pandas as pd
import numpy as np
from cleaning_pipeline import clean_agent_data

txns_raw = pd.read_csv("agent_transactions.csv")
agents_raw = pd.read_csv("agents.csv")

# Audit checks on raw data
dup_rows = int(txns_raw.duplicated(subset=["txn_id"]).sum())
non_std_type = int((~txns_raw["txn_type"].isin(["Deposit", "Withdrawal"])).sum())
slash_dates = int(txns_raw["txn_date"].astype(str).str.contains("/").sum())
missing_amt = int(pd.to_numeric(txns_raw["amount"], errors="coerce").isna().sum())
spike_250k = int((pd.to_numeric(txns_raw["amount"], errors="coerce") == 250000).sum())
unknown_agents = int((~txns_raw["agent_id"].isin(agents_raw["agent_id"])).sum())

audit_df = pd.DataFrame([
    {
        "Issue Found": "Duplicate txn_id rows (11 exact duplicates + 1 duplicate ID reassigned to AG099)",
        "Dimension Violated": "Uniqueness",
        "Affected Column": "txn_id",
        "Rows Affected": dup_rows,
    },
    {
        "Issue Found": "Inconsistent casing and trailing whitespace ('deposit', 'DEPOSIT ', 'withdrawal')",
        "Dimension Violated": "Consistency",
        "Affected Column": "txn_type",
        "Rows Affected": non_std_type,
    },
    {
        "Issue Found": "Slash-delimited dates (YYYY/MM/DD instead of ISO YYYY-MM-DD)",
        "Dimension Violated": "Validity",
        "Affected Column": "txn_date",
        "Rows Affected": slash_dates,
    },
    {
        "Issue Found": "Missing or 'N/A' amounts parsed as NaN",
        "Dimension Violated": "Completeness",
        "Affected Column": "amount",
        "Rows Affected": missing_amt,
    },
    {
        "Issue Found": "Extreme KES 250,000 spike values (~140x median)",
        "Dimension Violated": "Accuracy",
        "Affected Column": "amount",
        "Rows Affected": spike_250k,
    },
    {
        "Issue Found": "Transactions referencing unregistered agent_id 'AG099'",
        "Dimension Violated": "Integrity / Validity",
        "Affected Column": "agent_id",
        "Rows Affected": unknown_agents,
    },
])

display(audit_df)


,Issue Found,Dimension Violated,Affected Column,Rows Affected
0,Duplicate txn_id rows (11 exact duplicates + 1...,Uniqueness,txn_id,12
1,Inconsistent casing and trailing whitespace ('...,Consistency,txn_type,384
2,Slash-delimited dates (YYYY/MM/DD instead of I...,Validity,txn_date,5
3,Missing or 'N/A' amounts parsed as NaN,Completeness,amount,22
4,"Extreme KES 250,000 spike values (~140x median)",Accuracy,amount,5
5,Transactions referencing unregistered agent_id...,Integrity / Validity,agent_id,8


In [5]:
clean_df, cleaning_log = clean_agent_data(txns_raw, agents_raw)

print("Cleaning Log:", cleaning_log)
print("Unique txn_type values:", clean_df["txn_type"].unique().tolist())
print("Duplicate txn_id count:", clean_df["txn_id"].duplicated().sum())
print("Unknown agent rows post-merge:", (clean_df["county"] == "Unknown").sum())

os.makedirs("outputs", exist_ok=True)
clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False)


Cleaning Log: {'initial_rows': 612, 'duplicates_removed': 12, 'invalid_txn_types': 0, 'amounts_imputed': 20, 'imputation_median_kes': 1776.5, 'iqr_upper_fence': 5513.75, 'outliers_flagged': 53, 'unknown_agent_txns': 7, 'final_rows': 600}
Unique txn_type values: ['Deposit', 'Withdrawal']
Duplicate txn_id count: 0
Unknown agent rows post-merge: 7


In [6]:
# (a) County-level summary split by transaction type
county_summary = (
    clean_df.groupby(["county", "txn_type"], as_index=False)
    .agg(
        total_value=("amount", "sum"),
        txn_count=("txn_id", "count"),
        avg_value=("amount", "mean"),
    )
    .sort_values(["county", "txn_type"])
)
print("=== (a) County-Level Summary ===")
display(county_summary)

# (b) Pivot table of weekly transaction value by county
clean_df["week_start"] = clean_df["txn_date"].dt.to_period("W").apply(lambda p: p.start_time)
weekly_pivot = clean_df.pivot_table(
    index="week_start",
    columns="county",
    values="amount",
    aggfunc="sum",
    fill_value=0,
)
print("=== (b) Weekly Transaction Value by County (KES) ===")
display(weekly_pivot)


=== (a) County-Level Summary ===


,county,txn_type,total_value,txn_count,avg_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77050.5,27,2853.722222
2,Kisumu,Deposit,436420.5,74,5897.574324
3,Kisumu,Withdrawal,76374.5,37,2064.175676
4,Mombasa,Deposit,148986.0,72,2069.250000
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543590.0,118,4606.694915
7,Nairobi,Withdrawal,162036.0,62,2613.483871
8,Nakuru,Deposit,184890.0,60,3081.500000
9,Nakuru,Withdrawal,328594.5,39,8425.500000


=== (b) Weekly Transaction Value by County (KES) ===


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week_start,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2743.5,0.0
2026-03-02,18317.0,287933.0,285669.5,68967.0,56574.0,0.0
2026-03-09,267771.5,23186.0,30477.5,52860.5,28730.0,0.0
2026-03-16,22054.0,28125.0,25144.5,43107.5,39795.0,1639.0
2026-03-23,11504.0,27077.0,17050.0,41373.5,18998.5,0.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06,15030.0,15341.0,47092.5,56786.5,276609.0,9824.0
2026-04-13,12265.0,29416.0,26730.0,56432.5,26152.5,0.0
2026-04-20,24507.0,37439.0,16687.0,65187.0,30862.0,1877.0


In [7]:
'''
# Data Quality Report — Mobile Money Agent Network

### 1. Uniqueness
* **Finding:** 12 duplicate `txn_id` rows were identified in the 612-row raw extract (11 exact duplicates and 1 duplicate `TX00405` whose `agent_id` had been altered to `AG099`).
* **Resolution:** Deduplicated using `drop_duplicates(subset=["txn_id"], keep="first")`, removing **12 rows** and restoring **600 unique transactions** while preserving original agent attribution.

### 2. Consistency
* **Finding:** `txn_type` contained 5 inconsistent text variants (`deposit`, `DEPOSIT `, `withdrawal`, `Deposit`, `Withdrawal`) across 384 non-standard rows.
* **Resolution:** Applied `.str.strip().str.title()` to standardise all 600 records into `"Deposit"` and `"Withdrawal"`, verifying `0` missing or invalid entries.

### 3. Validity
* **Finding:** 5 records in `txn_date` used slash delimiters (`YYYY/MM/DD`) instead of ISO `YYYY-MM-DD`, and `amount` was stored as mixed strings (`"N/A"`, numeric strings).
* **Resolution:** Normalised `/` to `-` before parsing `txn_date` as `datetime64[ns]` and coerced `amount` to `float64`.

### 4. Completeness
* **Finding:** 20 unique transactions (22 before deduplication) had missing or `"N/A"` amounts.
* **Decision & Justification:** Rather than dropping 3.3% of valid transaction activity or distorting totals with a spike-inflated mean, missing amounts were **imputed with the dataset median (KES 1,776.50)** and flagged via `amount_imputed = True` so analysts can filter imputed rows during sensitivity checks.

### 5. Accuracy
* **Finding:** Using the IQR method (`Q1 = KES 1,105.75`, `Q3 = KES 2,869.00`, `Upper Fence = KES 5,513.75`), 53 rows were flagged with `is_outlier = True`. Notably, 5 transactions (`TX00307`, `TX00316`, `TX00425`, `TX00464`, `TX00595`) recorded exact **KES 250,000** values (~140× the median).
* **Recommendation:** These spikes likely reflect either **untagged agent float-rebalancing transfers** or **decimal-entry errors** (`2,500.00` entered without a decimal). They are retained and flagged (`is_outlier = True`) rather than deleted so total cash movement reconciles, escalated to Operations for audit, and excluded when calculating typical customer transaction averages.

### 6. Integrity
* **Finding:** 7 unique transactions (8 before deduplication) referenced `agent_id = "AG099"`, which does not exist in `agents.csv`.
* **Resolution:** Performed a **left join** onto `agents.csv` and assigned `county = "Unknown"` so valid transaction value is not silently lost while alerting Network Operations to register `AG099`.

### 7. Data Privacy & Protection (Kenya Data Protection Act, 2019)
If the raw transaction extract also contained **customer phone numbers**, both the **customer phone numbers** (direct personal identifiers) and **agent IDs combined with transaction timestamps and amounts** (indirect identifiers capable of singling out individuals) would constitute **personal data** under Section 2 of the **Kenya Data Protection Act (KDPA), 2019**. To comply with KDPA Section 25 principles of **data minimisation, purpose limitation, and storage limitation**, customer phone numbers must be **pseudonymised via salted SHA-256 hashing** prior to analytical ingestion, restricted via role-based access control, encrypted at rest and in transit, and excluded entirely from county-level and weekly summary outputs.
'''

'\n# Data Quality Report — Mobile Money Agent Network\n\n### 1. Uniqueness\n* **Finding:** 12 duplicate `txn_id` rows were identified in the 612-row raw extract (11 exact duplicates and 1 duplicate `TX00405` whose `agent_id` had been altered to `AG099`).\n* **Resolution:** Deduplicated using `drop_duplicates(subset=["txn_id"], keep="first")`, removing **12 rows** and restoring **600 unique transactions** while preserving original agent attribution.\n\n### 2. Consistency\n* **Finding:** `txn_type` contained 5 inconsistent text variants (`deposit`, `DEPOSIT `, `withdrawal`, `Deposit`, `Withdrawal`) across 384 non-standard rows.\n* **Resolution:** Applied `.str.strip().str.title()` to standardise all 600 records into `"Deposit"` and `"Withdrawal"`, verifying `0` missing or invalid entries.\n\n### 3. Validity\n* **Finding:** 5 records in `txn_date` used slash delimiters (`YYYY/MM/DD`) instead of ISO `YYYY-MM-DD`, and `amount` was stored as mixed strings (`"N/A"`, numeric strings).\n* **Res